# Control de calidad — SWE y profundidad de nieve (estaciones automáticas)

**UPLA / Glacies — Brief Report El Niño 2026**

Este cuaderno reproduce el pipeline de depuración usado en el estudio. Está pensado para **revisores**: todo el código y la documentación están en un solo archivo.

**Qué hace (por estación):**
1. **SWE** (`min`, `media`, `max`): recorte físico → detección de picos aislados (MAD) → suavizado (mediana móvil).
2. **SD** (solo reglas declaradas): p. ej. ceros abruptos en Laguna Negra.
3. **SWE vs SD** (opcional): alineación SWE=0 sin nieve; reglas coherencia almohada Quebrada Larga 2026.

**Datos:** rutas relativas a la raíz del repositorio `Data/Station_SWE/` y `Data/Station_SnowDepth/`. Los originales SWE se respaldan una vez en `Data/Station_SWE/_original_backup/`.

**Dependencias:** Python ≥ 3.10, `pandas`, `numpy`. Opcional: `matplotlib` para la celda de diagnóstico.

## 1. Configuración de rutas y estaciones

Edite `STATION_ID` para una sola estación, o deje `None` para procesar todas las entradas de `STATIONS`.

Para añadir una estación nueva, copie un bloque en `STATIONS` con `swe_filename`, `sd_filename`, `swe_max_mm` y parámetros MAD.

In [ ]:
from __future__ import annotations

import shutil
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Literal

import numpy as np
import pandas as pd

# --- Rutas (funciona ejecutando desde la raíz del repo o desde cleaning_script/) ---
_cwd = Path.cwd().resolve()
if (_cwd / "Data" / "Station_SWE").is_dir():
    PROJECT_ROOT = _cwd
elif (_cwd.parent / "Data" / "Station_SWE").is_dir():
    PROJECT_ROOT = _cwd.parent
else:
    PROJECT_ROOT = _cwd  # ajuste manual si clona solo este subfolder

SWE_DIR = PROJECT_ROOT / "Data" / "Station_SWE"
SD_DIR = PROJECT_ROOT / "Data" / "Station_SnowDepth"
BACKUP_DIR = SWE_DIR / "_original_backup"
OUTPUT_QC_DIR = PROJECT_ROOT / "cleaning_script" / "qc_outputs"
OUTPUT_QC_DIR.mkdir(parents=True, exist_ok=True)

# --- Ejecución ---
STATION_ID: str | None = None  # p. ej. "05703009"; None = todas
WRITE_FILES = True  # False = solo informe en memoria / CSV en qc_outputs
MAKE_BACKUP = True

SWE_DEFAULT_CLEAN = {
    "fwindow": 3,
    "k": 4.0,
    "abs_thr_mm": 100.0,
    "smooth_window": 3,
}

SWE_ALIGN = {
    "min_sd_cm": 2.0,
    "gate_days": 7,
    "infill_max_gap_days": 7,
    "infill_min_context_mm": 15.0,
}

QUEBRADA_SWE_QC = {
    "years": (2026,),
    "swe_limit_mm": 3500.0,
    "sd_min_cm": 50.0,
    "ratio_min": 0.05,
    "ratio_max": 12.5,
    "shallow_sd_min_cm": 5.0,
    "shallow_sd_max_cm": 10.0,
    "shallow_swe_max_mm": 40.0,
}

LAGUNA_SD_QC = {
    "prev_depth_min_cm": 50.0,
    "ffill_limit_days": 3,
}


@dataclass
class StationProfile:
    station_id: str
    name: str
    swe_filename: str
    sd_filename: str
    swe_max_mm: float = 3500.0
    swe_clean: dict[str, Any] = field(default_factory=lambda: dict(SWE_DEFAULT_CLEAN))
    sd_rule: Literal["none", "laguna_abrupt_zero"] = "none"
    swe_rule: Literal["none", "quebrada_pillow_vs_sd"] = "none"
    apply_swe_sd_align: bool = True


STATIONS: dict[str, StationProfile] = {
    "04520006": StationProfile(
        "04520006", "Quebrada Larga",
        "QUEBRADA_LARGA_SWE.csv", "QUEBRADA_LARGA_SD.csv",
        swe_max_mm=3500.0, swe_rule="quebrada_pillow_vs_sd",
    ),
    "05401007": StationProfile(
        "05401007", "Portillo",
        "PORTILLO_SWE.csv", "PORTILLO_SD.csv",
        swe_max_mm=1200.0,
        swe_clean={**SWE_DEFAULT_CLEAN, "k": 3.5, "abs_thr_mm": 75.0, "smooth_window": 9},
    ),
    "05703009": StationProfile(
        "05703009", "Laguna Negra",
        "LAGUNA_NEGRA_SWE.csv", "LAGUNA_NEGRA_SD.csv",
        swe_max_mm=3000.0, sd_rule="laguna_abrupt_zero",
    ),
    "06020001": StationProfile(
        "06020001", "Termas del Flaco",
        "TERMAS_DEL_FLACO_SWE.csv", "TERMAS_DEL_FLACO_SD.csv",
        swe_max_mm=1200.0,
    ),
    "07301000": StationProfile(
        "07301000", "Lo Aguirre",
        "LO_AGUIRRE_SWE.csv", "LO_AGUIRRE_SD.csv",
        swe_max_mm=1200.0,
    ),
}

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"Estaciones configuradas: {len(STATIONS)}")


## 2. Funciones modulares — SWE (MAD + suavizado)

Parámetros por defecto en `SWE_DEFAULT_CLEAN`; cada estación puede sobreescribir `swe_clean` en `STATIONS`.

In [ ]:
SWE_COLS = ("min", "media", "max")


def detect_spikes_mad(
    data: pd.DataFrame,
    cols: list[str],
    *,
    fwindow: int = 3,
    k: float = 4.0,
    abs_thr: float = 100.0,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    df_clean = data.copy()
    flags = pd.DataFrame(0, index=df_clean.index, columns=cols, dtype="int8")
    win_size = 2 * fwindow + 1
    for col in cols:
        s = df_clean[col].astype(float)
        med = s.rolling(win_size, center=True).median()
        mad = (s - med).abs().rolling(win_size, center=True).median()
        mad_safe = mad.replace(0, np.nan)
        z = (s - med).abs() / mad_safe
        prev_diff = (s - s.shift(1)).abs()
        next_diff = (s - s.shift(-1)).abs()
        neigh_diff = (s.shift(1) - s.shift(-1)).abs()
        safe_prev = prev_diff.fillna(np.inf)
        safe_next = next_diff.fillna(np.inf)
        spike_mad = (
            (mad_safe > 0)
            & (z > k)
            & (safe_prev > k * mad_safe)
            & (safe_next > k * mad_safe)
            & (neigh_diff.fillna(0) < k * mad_safe)
        )
        spike_abs = (prev_diff > abs_thr) & (next_diff > abs_thr)
        spike = spike_mad | spike_abs
        df_clean.loc[spike, col] = np.nan
        flags.loc[spike, col] = 1
    return df_clean, flags


def clip_swe_nonnegative(data: pd.DataFrame, cols: list[str] | None = None) -> pd.DataFrame:
    out = data.copy()
    use = list(cols or SWE_COLS)
    for col in use:
        if col not in out.columns:
            continue
        out[col] = pd.to_numeric(out[col], errors="coerce").clip(lower=0)
    return out


def cap_swe_physical(data: pd.DataFrame, cols: list[str], *, max_mm: float) -> pd.DataFrame:
    out = clip_swe_nonnegative(data, cols)
    for col in cols:
        s = out[col].astype(float)
        out.loc[s.gt(max_mm), col] = np.nan
    return out


def smooth_swe_series(
    data: pd.DataFrame,
    cols: list[str],
    *,
    window: int = 3,
    interpolate_limit: int = 2,
) -> pd.DataFrame:
    out = data.copy()
    for col in cols:
        s = out[col].astype(float)
        s = s.interpolate(limit=interpolate_limit, limit_direction="both")
        s = s.rolling(window, center=True, min_periods=1).median()
        out[col] = s
    return out


def clean_swe_dataframe(
    df: pd.DataFrame,
    *,
    max_mm: float,
    fwindow: int = 3,
    k: float = 4.0,
    abs_thr_mm: float = 100.0,
    smooth_window: int = 3,
) -> tuple[pd.DataFrame, dict[str, int]]:
    work = df.copy()
    for col in SWE_COLS:
        if col not in work.columns:
            raise ValueError(f"Missing column {col}")
    present = [c for c in SWE_COLS if work[c].notna().any()]
    if not present:
        return work, {c: 0 for c in SWE_COLS}
    work = cap_swe_physical(work, present, max_mm=max_mm)
    cleaned, flags = detect_spikes_mad(
        work, present, fwindow=fwindow, k=k, abs_thr=abs_thr_mm
    )
    cleaned = smooth_swe_series(cleaned, present, window=smooth_window)
    cleaned = clip_swe_nonnegative(cleaned, present)
    counts = {col: int(flags[col].sum()) if col in flags.columns else 0 for col in SWE_COLS}
    return cleaned, counts


## 3. Funciones modulares — SD y coherencia SWE–SD

In [ ]:
def qc_laguna_abrupt_zeros(
    sd: pd.DataFrame,
    *,
    prev_depth_min_cm: float,
    ffill_limit_days: int,
) -> pd.DataFrame:
    out = sd.sort_values("date").copy()
    s = out["SD_mean_clean"]
    prev_valid = s.ffill(limit=ffill_limit_days).shift(1)
    bad = s.eq(0) & prev_valid.gt(prev_depth_min_cm)
    out.loc[bad, "SD_mean_clean"] = pd.NA
    raw_bad = bad & out["SD_mean_raw"].eq(0)
    out.loc[raw_bad, "SD_mean_raw"] = pd.NA
    return out


def qc_quebrada_pillow_swe(
    swe: pd.DataFrame,
    sd: pd.DataFrame,
    *,
    years: tuple[int, ...],
    swe_limit_mm: float,
    sd_min_cm: float,
    ratio_min: float,
    ratio_max: float,
    shallow_sd_min_cm: float,
    shallow_sd_max_cm: float,
    shallow_swe_max_mm: float,
) -> pd.DataFrame:
    out = swe.sort_values("date").copy()
    merged = out.merge(sd[["date", "SD_mean_clean"]], on="date", how="left")
    year_mask = merged["date"].dt.year.isin(years)
    has_swe = merged["media"].notna()
    sd_v = merged["SD_mean_clean"]
    swe_v = merged["media"]
    ratio = swe_v / sd_v
    bad = year_mask & has_swe & (
        (swe_v < 0)
        | (swe_v.abs() > swe_limit_mm)
        | sd_v.isna()
        | sd_v.lt(sd_min_cm)
        | (sd_v.ge(10) & (ratio.lt(ratio_min) | ratio.gt(ratio_max)))
        | (
            sd_v.ge(shallow_sd_min_cm)
            & sd_v.lt(shallow_sd_max_cm)
            & swe_v.gt(shallow_swe_max_mm)
        )
    )
    out.loc[out["date"].isin(merged.loc[bad, "date"]), "media"] = pd.NA
    return out


def _infill_short_swe_gaps(values: np.ndarray, *, max_gap: int, min_context_mm: float) -> np.ndarray:
    v = values.astype(float, copy=True)
    n = len(v)
    i = 0
    while i < n:
        if not (v[i] == 0 or np.isnan(v[i])):
            i += 1
            continue
        j = i
        while j < n and (v[j] == 0 or np.isnan(v[j])):
            j += 1
        run_len = j - i
        left = v[i - 1] if i > 0 else np.nan
        right = v[j] if j < n else np.nan
        if (
            run_len <= max_gap
            and np.isfinite(left)
            and np.isfinite(right)
            and left >= min_context_mm
            and right >= min_context_mm
        ):
            for k in range(i, j):
                t = (k - i + 1) / (run_len + 1)
                v[k] = left + t * (right - left)
        i = j if j > i else i + 1
    return v


def align_swe_with_sd(
    swe: pd.DataFrame,
    sd: pd.DataFrame,
    *,
    sd_col: str = "SD_mean_clean",
    swe_cols: tuple[str, ...] = SWE_COLS,
    min_sd_cm: float,
    gate_days: int,
    infill_max_gap: int,
    infill_min_context_mm: float,
) -> pd.DataFrame:
    out = swe.sort_values("date").copy()
    sd_sorted = sd.sort_values("date")
    merged = out.merge(sd_sorted[["date", sd_col]], on="date", how="left")
    merged["water_year"] = np.where(
        merged["date"].dt.month >= 4,
        merged["date"].dt.year,
        merged["date"].dt.year - 1,
    )
    present = [c for c in swe_cols if c in out.columns]
    for _, idx in merged.groupby("water_year").groups.items():
        block = merged.loc[idx]
        sd_gate = block[sd_col].rolling(gate_days, center=True, min_periods=1).max()
        no_snow = sd_gate.isna() | sd_gate.lt(min_sd_cm)
        if no_snow.any():
            zero_idx = block.index[no_snow.to_numpy()]
            for col in present:
                out.loc[zero_idx, col] = 0.0
        for col in present:
            segment = out.loc[block.index, col].to_numpy()
            filled = _infill_short_swe_gaps(
                segment,
                max_gap=infill_max_gap,
                min_context_mm=infill_min_context_mm,
            )
            out.loc[block.index, col] = filled
    return out


## 4. Orquestación por estación

Orden: respaldo SWE → limpieza MAD → QC SD (si aplica) → alineación SWE–SD → reglas SWE extra (si aplica).

In [ ]:
def backup_swe_once(path: Path) -> Path:
    BACKUP_DIR.mkdir(parents=True, exist_ok=True)
    dest = BACKUP_DIR / path.name
    if not dest.exists() and path.exists():
        shutil.copy2(path, dest)
        print(f"  backup → {dest}")
    return dest if dest.exists() else path


def load_sd(profile: StationProfile) -> pd.DataFrame | None:
    path = SD_DIR / profile.sd_filename
    if not path.exists():
        return None
    return pd.read_csv(path, parse_dates=["date"])


def run_station_qc(profile: StationProfile) -> dict[str, Any]:
    swe_path = SWE_DIR / profile.swe_filename
    report: dict[str, Any] = {"station_id": profile.station_id, "name": profile.name}

    if not swe_path.exists():
        report["error"] = f"Missing {swe_path}"
        return report

    if MAKE_BACKUP:
        backup_swe_once(swe_path)
    src = BACKUP_DIR / swe_path.name
    if not src.exists():
        src = swe_path
    swe_raw = pd.read_csv(src, parse_dates=["date"])

    kwargs = {**SWE_DEFAULT_CLEAN, **profile.swe_clean}
    swe_clean, spikes = clean_swe_dataframe(swe_raw, max_mm=profile.swe_max_mm, **kwargs)
    report["spikes"] = spikes

    sd = load_sd(profile)
    sd_changes = 0
    if sd is not None and profile.sd_rule == "laguna_abrupt_zero":
        before = sd["SD_mean_clean"].eq(0).sum()
        sd = qc_laguna_abrupt_zeros(sd, **LAGUNA_SD_QC)
        sd_changes = int(before - sd["SD_mean_clean"].eq(0).sum())
        report["laguna_sd_zeros_removed"] = sd_changes

    if sd is not None and profile.apply_swe_sd_align:
        swe_clean = align_swe_with_sd(swe_clean, sd, **SWE_ALIGN)

    quebrada_nulled = 0
    if sd is not None and profile.swe_rule == "quebrada_pillow_vs_sd":
        n_before = swe_clean["media"].notna().sum()
        swe_clean = qc_quebrada_pillow_swe(swe_clean, sd, **QUEBRADA_SWE_QC)
        quebrada_nulled = int(n_before - swe_clean["media"].notna().sum())
        report["quebrada_media_nulled"] = quebrada_nulled

    out_swe = OUTPUT_QC_DIR / profile.swe_filename
    out_sd = OUTPUT_QC_DIR / profile.sd_filename
    swe_clean.to_csv(out_swe, index=False, date_format="%Y-%m-%d")
    report["qc_output_swe"] = str(out_swe)
    if sd is not None:
        sd.to_csv(out_sd, index=False, date_format="%Y-%m-%d")
        report["qc_output_sd"] = str(out_sd)

    if WRITE_FILES:
        swe_clean.to_csv(swe_path, index=False, date_format="%Y-%m-%d")
        if sd is not None and profile.sd_rule == "laguna_abrupt_zero":
            (SD_DIR / profile.sd_filename).parent.mkdir(parents=True, exist_ok=True)
            sd.to_csv(SD_DIR / profile.sd_filename, index=False, date_format="%Y-%m-%d")
        report["wrote_data"] = True
    else:
        report["wrote_data"] = False

    return report


def run_all(selected: str | None = STATION_ID) -> pd.DataFrame:
    ids = [selected] if selected else list(STATIONS)
    rows = []
    for sid in ids:
        if sid not in STATIONS:
            raise KeyError(f"Unknown station {sid}")
        prof = STATIONS[sid]
        print(f"\n=== {prof.name} ({sid}) ===")
        rows.append(run_station_qc(prof))
    return pd.DataFrame(rows)

summary = run_all(STATION_ID)
try:
    from IPython.display import display

    display(summary)
except ImportError:
    print(summary.to_string(index=False))


## 5. Diagnóstico visual (opcional)

Compara SWE `media` cruda (respaldo) vs limpia para una estación. Requiere `matplotlib`.

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    print("Instale matplotlib para esta celda: pip install matplotlib")
else:
    diag_id = STATION_ID or "05401007"
    prof = STATIONS[diag_id]
    raw_path = BACKUP_DIR / prof.swe_filename
    if not raw_path.exists():
        raw_path = SWE_DIR / prof.swe_filename
    raw = pd.read_csv(raw_path, parse_dates=["date"])
    clean = pd.read_csv(OUTPUT_QC_DIR / prof.swe_filename, parse_dates=["date"])
    fig, ax = plt.subplots(figsize=(10, 3.5))
    ax.plot(raw["date"], raw["media"], color="#ccc", lw=0.8, label="media (raw / backup)")
    ax.plot(clean["date"], clean["media"], color="#c45c26", lw=1.0, label="media (QC)")
    ax.set_title(f"{prof.name} — SWE media")
    ax.set_ylabel("mm")
    ax.legend(loc="upper right", fontsize=8)
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()


## 6. Referencia rápida (manuscrito)

| Paso | Descripción |
|------|-------------|
| SWE | Clip ≥ 0, null > max estación, picos MAD (ventana 7 d), mediana móvil |
| Laguna Negra SD | SD=0 tras pack > 50 cm (ffill 3 d) → missing |
| Quebrada Larga 2026 | SWE almohada vs SD mismo día (tabla en `README.md` / `QC_PARAMETERS.csv`) |
| Alineación | SWE=0 si max móvil 7 d de SD < 2 cm; infill lineal breve |

Scripts equivalentes en la carpeta `cleaning_script/`: `run_all_qc.py`.